# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [1]:
# Cell 1: Installation & Imports

# 1. Install missing libraries
import subprocess
import sys
!pip install scikit-multilearn -q


def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

try:
    import evaluate
except ImportError:
    print("Installing 'evaluate'...")
    install("evaluate")
    install("accelerate")

# 2. Imports
import os
import json

# Check if running on Modal
try:
    import modal
    RUNNING_ON_MODAL = True
except ImportError:
    RUNNING_ON_MODAL = False
    print("Not running on Modal. Ensure you have libraries installed locally.")

import torch
import numpy as np
import evaluate
from transformers import (
    DistilBertTokenizer, 
    DistilBertForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

print("All libraries loaded successfully.")


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Installing 'evaluate'...



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


All libraries loaded successfully.


In [2]:
# In a Modal Notebook, we usually define the dependencies in the image, 
# but you can also run pip installs in the first cell if needed.

import os
import json
import modal

# Check if running on Modal
try:
    import modal
    RUNNING_ON_MODAL = True
except ImportError:
    RUNNING_ON_MODAL = False
    print("Not running on Modal. Ensure you have libraries installed locally.")

import torch
import numpy as np
import evaluate
from transformers import (
    DistilBertTokenizer, 
    DistilBertForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

In [3]:
# --- MODAL CONFIGURATION ---
# Your volume is mounted at /mnt/data
MOUNT_PATH = "/mnt/datanew" 
# DATA_FILE = "training_data.jsonl"
DATA_FILE = "nlc_dataset_new.jsonl"

# We will save the model back to this same volume
MODEL_SAVE_PATH = f"{MOUNT_PATH}/fine_tuned_model"

# Create directory if it doesn't exist (optional safety check)
import os
if not os.path.exists(MOUNT_PATH):
    print(f"Warning: Mount path {MOUNT_PATH} does not exist. Check Modal Volume mount.")

print(f"Data path set to: {os.path.join(MOUNT_PATH, DATA_FILE)}")
print(f"Model will be saved to: {MODEL_SAVE_PATH}")

Data path set to: /mnt/datanew/nlc_dataset_new.jsonl
Model will be saved to: /mnt/datanew/fine_tuned_model


In [4]:
from skmultilearn.model_selection import IterativeStratification
# Label Definition (Alphabetical Order)
LABELS = [
    'adventure_negative', 'adventure_positive', 'beaches_negative', 'beaches_positive',
    'food_negative', 'food_positive', 'historical_negative', 'historical_positive',
    'nature_negative', 'nature_positive', 'nightlife_negative', 'nightlife_positive',
    'religious_negative', 'religious_positive', 'shopping_negative', 'shopping_positive'
]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

def load_and_process_data(mount_path, filename):
    """Loads JSONL and formats labels for multi-label classification."""
    texts = []
    labels = []
    
    full_path = os.path.join(mount_path, filename)
    
    print(f"Loading data from: {full_path}")
    
    try:
        with open(full_path, 'r') as f:
            for line in f:
                entry = json.loads(line)
                texts.append(entry['text'])
                
                # Convert dict labels to list of floats in correct order
                label_vector = [float(entry['labels'][key]) for key in LABELS]
                labels.append(label_vector)
                
        return texts, labels
    except FileNotFoundError:
        print(f"ERROR: File not found at {full_path}")
        return [], []

# Load the data
texts, labels = load_and_process_data(MOUNT_PATH, DATA_FILE)

# FILTER ZERO-LABEL SAMPLES
filtered_texts = []
filtered_labels = []
for text, label in zip(texts, labels):
    if sum(label) > 0:  # Keep only samples with at least 1 label
        filtered_texts.append(text)
        filtered_labels.append(label)
texts = filtered_texts
labels = filtered_labels
print(f"After filtering: {len(texts)} samples")


if len(texts) == 0:
    print("No data loaded! Please check the file path.")
else:
    print(f"Loaded {len(texts)} samples.")
    
    # Split Data
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        texts, labels, test_size=0.2, random_state=42
    )
    
    # label_array = np.array(labels)
    # splitter = IterativeStratification(n_labels=16, n_samples=len(labels), test_size=0.2, random_state=42)
    # train_idx, val_idx = next(splitter.split(label_array, label_array))

    # train_texts = [texts[i] for i in train_idx]
    # val_texts = [texts[i] for i in val_idx]
    # train_labels = [labels[i] for i in train_idx]
    # val_labels = [labels[i] for i in val_idx]

    # label_array = np.array(labels)

    # splitter = IterativeStratification(n_splits=2,
    # order=1,
    # sample_distribution_per_fold=[0.8, 0.2],
    # random_state=42
    # )

    # train_idx, val_idx = next(splitter.split(label_array, label_array))

    # train_texts = [texts[i] for i in train_idx]
    # val_texts = [texts[i] for i in val_idx]
    # train_labels = [labels[i] for i in train_idx]
    # val_labels = [labels[i] for i in val_idx]

    

    # Tokenize
    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

    train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
    val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

    # Create Dataset Class
    class GoaTravelDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels

        def __getitem__(self, idx):
            item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float) 
            return item

        def __len__(self):
            return len(self.labels)

    train_dataset = GoaTravelDataset(train_encodings, train_labels)
    val_dataset = GoaTravelDataset(val_encodings, val_labels)
    print("Datasets ready.")

Loading data from: /mnt/datanew/nlc_dataset_new.jsonl
After filtering: 3840 samples
Loaded 3840 samples.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Datasets ready.


In [5]:
# Cell 4: Metrics & Model Initialization

# Load Metrics
# We use 'evaluate' library for standard metrics
f1_metric = evaluate.load("f1", "multilabel")
accuracy_metric = evaluate.load("accuracy")

# Calculate inverse frequency weights
label_counts = np.array([
    sum(1 for item in labels if item[i] == 1) for i in range(len(LABELS))
])
class_weights = (len(labels) / (len(LABELS) * label_counts))
class_weights = class_weights / class_weights.mean()
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)


def compute_metrics(eval_pred):
    """
    Computes F1, Precision, and Recall for multi-label classification.
    """
    predictions, labels = eval_pred
    
    # 1. Apply Sigmoid to logits to get probabilities (0 to 1)
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    
    # 2. Apply threshold (0.5) to convert probabilities to binary (0 or 1)
    preds = (probs > 0.5).int().numpy()
    labels = labels.astype(int)
    
    # 3. Calculate Metrics
    # Micro F1: Calculates metrics globally by counting total true positives, 
    # false negatives and false positives. Good for imbalanced data.
    f1_micro = f1_metric.compute(predictions=preds, references=labels, average='micro')
    
    # Sample F1: Calculate metrics for each instance and find their average.
    f1_sample = f1_metric.compute(predictions=preds, references=labels, average='samples')
    
    return {
        'f1_micro': f1_micro['f1'],
        'f1_sample': f1_sample['f1']
    }

class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if self.class_weights is not None:
            pos_weight = self.class_weights.to(logits.device)
            loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        else:
            loss_fct = torch.nn.BCEWithLogitsLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Initialize Model
print("Initializing DistilBERT model...")
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    problem_type="multi_label_classification", # Crucial for 16 labels
    num_labels=len(LABELS)                     # 16 labels
)

print("Model initialized successfully.")

Initializing DistilBERT model...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model initialized successfully.


In [8]:
# Cell 5: Training (Corrected)
from transformers import EarlyStoppingCallback
# Define Training Arguments
training_args = TrainingArguments(
    output_dir="/tmp/results",           # Temporary output for checkpoints during run
    num_train_epochs=15,                 # 3 epochs is standard for fine-tuning
    per_device_train_batch_size=16,      # Batch size (fits on T4/A10G GPUs)
    per_device_eval_batch_size=16,
    learning_rate=3e-5,                  # Standard learning rate for BERT
    weight_decay=0.01,                   # Regularization
    eval_strategy="epoch",         # CHANGED: evaluation_strategy -> eval_strategy
    save_strategy="epoch",               # Save model at end of each epoch
    load_best_model_at_end=True,         # Load the best model automatically
    logging_dir="/tmp/logs",
    logging_steps=10,
    report_to="none",                    # Disable external reporting (wandb, etc.)
    metric_for_best_model="f1_micro"     # Use F1 Micro to decide best model
)

# Initialize Trainer
trainer = WeightedLossTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Start Training
print("Starting training...")
trainer.train()

print("Training complete.")

/tmp/ipykernel_123/1326891095.py:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedLossTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Starting training...


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Sample
1,0.269000,0.245304,0.096100,0.034142
2,0.121400,0.107673,0.870303,0.879303
3,0.065500,0.055294,0.958143,0.960587
4,0.030900,0.033248,0.984340,0.987959
5,0.027200,0.023088,0.989231,0.991280
6,0.015100,0.018928,0.987737,0.989267
7,0.016200,0.016494,0.987755,0.989528
8,0.012100,0.014749,0.986657,0.989311


Training complete.


In [9]:
 #Save Model to Volume

# Save the fine-tuned model and tokenizer to the persistent Modal Volume path
print(f"Saving model to: {MODEL_SAVE_PATH}")

trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

print("Model and tokenizer saved successfully.")

Saving model to: /mnt/datanew/fine_tuned_model
Model and tokenizer saved successfully.


In [10]:
"""
Generate all visualization diagrams for the travel interest classification model.
Run this script to create all necessary images for model explanation.
"""

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import torch
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
import os

# Create output directory if it doesn't exist
OUTPUT_DIR = '/mnt/user-data/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set style for better looking plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

# Labels from your model
LABELS = [
    'adventure_negative', 'adventure_positive', 'beaches_negative', 'beaches_positive',
    'food_negative', 'food_positive', 'historical_negative', 'historical_positive',
    'nature_negative', 'nature_positive', 'nightlife_negative', 'nightlife_positive',
    'religious_negative', 'religious_positive', 'shopping_negative', 'shopping_positive'
]

LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

# Your training results
TRAINING_DATA = {
    'epochs': list(range(1, 21)),
    'train_loss': [0.2368, 0.1113, 0.0555, 0.0301, 0.0241, 0.0148, 0.0111, 0.0080, 
                   0.0101, 0.0151, 0.0057, 0.0050, 0.0036, 0.0033, 0.0028, 0.0034, 
                   0.0029, 0.0024, 0.0024, 0.0022],
    'val_loss': [0.2138, 0.0981, 0.0483, 0.0329, 0.0249, 0.0216, 0.0208, 0.0195, 
                 0.0190, 0.0186, 0.0187, 0.0183, 0.0196, 0.0204, 0.0196, 0.0209, 
                 0.0210, 0.0210, 0.0211, 0.0210],
    'f1_micro': [0.1859, 0.8369, 0.9606, 0.9674, 0.9712, 0.9749, 0.9769, 0.9780, 
                 0.9746, 0.9780, 0.9772, 0.9772, 0.9754, 0.9758, 0.9754, 0.9765, 
                 0.9765, 0.9765, 0.9761, 0.9761],
    'f1_sample': [0.0913, 0.7396, 0.8749, 0.8812, 0.8847, 0.8868, 0.8891, 0.8892,
                  0.8871, 0.8896, 0.8891, 0.8894, 0.8876, 0.8892, 0.8879, 0.8892,
                  0.8892, 0.8892, 0.8890, 0.8890]
}


def diagram_1_training_curves():
    """
    Create training and validation loss curves with F1 scores.
    Shows model learning progress and overfitting detection.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = TRAINING_DATA['epochs']
    
    # Plot 1: Loss curves
    ax1.plot(epochs, TRAINING_DATA['train_loss'], 'b-o', label='Training Loss', 
             linewidth=2, markersize=5)
    ax1.plot(epochs, TRAINING_DATA['val_loss'], 'r-s', label='Validation Loss', 
             linewidth=2, markersize=5)
    ax1.axvline(x=8, color='green', linestyle='--', linewidth=2, 
                label='Best Model (Epoch 8)', alpha=0.7)
    ax1.fill_between([8, 20], 0, 0.25, alpha=0.1, color='red', 
                     label='Overfitting Zone')
    ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Loss', fontsize=12, fontweight='bold')
    ax1.set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 21)
    
    # Plot 2: F1 Scores
    ax2.plot(epochs, TRAINING_DATA['f1_micro'], 'g-o', label='F1 Micro', 
             linewidth=2, markersize=5)
    ax2.plot(epochs, TRAINING_DATA['f1_sample'], 'm-^', label='F1 Sample', 
             linewidth=2, markersize=5)
    ax2.axvline(x=8, color='green', linestyle='--', linewidth=2, 
                label='Best Model', alpha=0.7)
    ax2.axhline(y=0.95, color='orange', linestyle=':', linewidth=1.5, 
                label='95% Target', alpha=0.7)
    ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax2.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
    ax2.set_title('Model Performance (F1 Scores)', fontsize=14, fontweight='bold')
    ax2.legend(loc='lower right')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, 21)
    ax2.set_ylim(0, 1.05)
    
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/1_training_curves.png', dpi=300, bbox_inches='tight')
    print("✓ Created: 1_training_curves.png")
    plt.close()


def diagram_2_label_distribution(data_path):
    """
    Show distribution of labels in the dataset.
    Helps identify class imbalance issues.
    """
    # Load data
    data = []
    with open(data_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    
    # Count label occurrences
    label_counts = Counter()
    for item in data:
        for label in item['labels']:
            label_counts[label] += 1
    
    # Sort labels for better visualization
    sorted_labels = sorted(LABELS)
    counts = [label_counts[label] for label in sorted_labels]
    
    # Create colors: green for positive, red for negative
    colors = ['green' if 'positive' in label else 'red' for label in sorted_labels]
    
    # Create plot
    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.bar(range(len(sorted_labels)), counts, color=colors, alpha=0.7, 
                  edgecolor='black', linewidth=1)
    
    # Add value labels on bars
    for i, (bar, count) in enumerate(zip(bars, counts)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 5,
                f'{count}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Format labels
    clean_labels = [label.replace('_', ' ').title() for label in sorted_labels]
    ax.set_xticks(range(len(sorted_labels)))
    ax.set_xticklabels(clean_labels, rotation=45, ha='right', fontsize=10)
    ax.set_xlabel('Interest Category', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Examples', fontsize=12, fontweight='bold')
    ax.set_title('Distribution of Labels in Training Dataset', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='green', alpha=0.7, label='Positive Interest'),
        Patch(facecolor='red', alpha=0.7, label='Negative Interest')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    # Add statistics
    total = len(data)
    avg_labels_per_example = sum(len(item['labels']) for item in data) / total
    stats_text = f"Total Examples: {total}\nAvg Labels/Example: {avg_labels_per_example:.2f}"
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/2_label_distribution.png', dpi=300, bbox_inches='tight')
    print("✓ Created: 2_label_distribution.png")
    print(f"  Total examples: {total}")
    print(f"  Most common: {label_counts.most_common(1)[0]}")
    print(f"  Least common: {label_counts.most_common()[-1]}")
    plt.close()


def diagram_3_per_category_performance(model_path, data_path, is_validation=True):
    """
    Show F1 score for each category.
    This is the most important diagram - shows which categories work well.
    """
    # Load model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)
    tokenizer = DistilBertTokenizer.from_pretrained(model_path)
    model.eval()
    
    # Load data
    data = []
    with open(data_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    
    # If you want to test on validation set, split the data
    # For now, we'll use all data for demonstration
    if is_validation:
        # Take last 20% as validation
        split_idx = int(len(data) * 0.8)
        data = data[split_idx:]
    
    print(f"Evaluating on {len(data)} examples...")
    
    # Get predictions
    all_preds = []
    all_labels = []
    
    for item in data:
        text = item['text']
        true_labels = [1 if label in item['labels'] else 0 for label in LABELS]
        
        inputs = tokenizer(text, return_tensors="pt", truncation=True, 
                          padding=True, max_length=128).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.sigmoid(logits).squeeze().cpu().numpy()
        
        preds = (probs > 0.5).astype(int)
        all_preds.append(preds)
        all_labels.append(true_labels)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Calculate per-category metrics
    category_metrics = []
    for i, label in enumerate(LABELS):
        pred_col = all_preds[:, i]
        true_col = all_labels[:, i]
        
        tp = ((pred_col == 1) & (true_col == 1)).sum()
        fp = ((pred_col == 1) & (true_col == 0)).sum()
        fn = ((pred_col == 0) & (true_col == 1)).sum()
        tn = ((pred_col == 0) & (true_col == 0)).sum()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        support = true_col.sum()
        
        category_metrics.append({
            'label': label,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'support': support
        })
    
    # Create plot
    fig, ax = plt.subplots(figsize=(14, 7))
    
    labels = [m['label'] for m in category_metrics]
    f1_scores = [m['f1'] for m in category_metrics]
    colors = ['green' if 'positive' in label else 'red' for label in labels]
    
    bars = ax.bar(range(len(labels)), f1_scores, color=colors, alpha=0.7, 
                  edgecolor='black', linewidth=1)
    
    # Add threshold lines
    ax.axhline(y=0.95, color='orange', linestyle='--', linewidth=2, 
               label='95% Threshold', alpha=0.7)
    ax.axhline(y=0.90, color='yellow', linestyle=':', linewidth=1.5, 
               label='90% Threshold', alpha=0.5)
    
    # Add value labels
    for i, (bar, f1) in enumerate(zip(bars, f1_scores)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{f1:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Format
    clean_labels = [label.replace('_', ' ').title() for label in labels]
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(clean_labels, rotation=45, ha='right', fontsize=10)
    ax.set_xlabel('Interest Category', fontsize=12, fontweight='bold')
    ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
    ax.set_title('Per-Category Model Performance', fontsize=14, fontweight='bold')
    ax.set_ylim(0.85, 1.02)
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(loc='lower right')
    
    # Add overall statistics
    avg_f1 = np.mean(f1_scores)
    min_f1 = min(f1_scores)
    stats_text = f"Average F1: {avg_f1:.3f}\nMin F1: {min_f1:.3f}\nCategories > 95%: {sum(f > 0.95 for f in f1_scores)}/16"
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/3_per_category_performance.png', dpi=300, bbox_inches='tight')
    print("✓ Created: 3_per_category_performance.png")
    print(f"  Average F1: {avg_f1:.3f}")
    print(f"  Categories with F1 > 95%: {sum(f > 0.95 for f in f1_scores)}/16")
    
    # Print detailed results
    print("\nDetailed per-category results:")
    for m in category_metrics:
        print(f"  {m['label']:25s} - F1: {m['f1']:.3f}, Precision: {m['precision']:.3f}, Recall: {m['recall']:.3f}, Support: {int(m['support'])}")
    
    plt.close()
    return category_metrics


def diagram_4_example_predictions(model_path):
    """
    Show real example predictions with confidence scores.
    Demonstrates how the model actually works.
    """
    # Load model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)
    tokenizer = DistilBertTokenizer.from_pretrained(model_path)
    model.eval()
    
    # Test examples
    test_examples = [
        "I love exploring ancient temples and historical sites",
        "I enjoy beach parties and nightlife but hate museums",
        "Looking for adventure activities like hiking and camping",
        "I prefer quiet nature walks, not interested in shopping or nightlife",
        "I want to try local food and visit markets",
        "Beach resorts are boring, I prefer mountain trekking"
    ]
    
    # Create figure
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.4, wspace=0.3)
    
    for idx, text in enumerate(test_examples):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        
        # Get predictions
        inputs = tokenizer(text, return_tensors="pt", truncation=True, 
                          padding=True, max_length=128).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.sigmoid(logits).squeeze().cpu().numpy()
        
        # Get top 5 predictions
        top_indices = probs.argsort()[-5:][::-1]
        top_labels = [LABELS[i].replace('_', ' ').title() for i in top_indices]
        top_probs = [probs[i] for i in top_indices]
        top_colors = ['green' if 'Positive' in label else 'red' for label in top_labels]
        
        # Plot
        bars = ax.barh(range(5), top_probs, color=top_colors, alpha=0.7, 
                       edgecolor='black', linewidth=1)
        ax.set_yticks(range(5))
        ax.set_yticklabels(top_labels, fontsize=9)
        ax.set_xlim(0, 1)
        ax.set_xlabel('Confidence Score', fontsize=9)
        ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1, alpha=0.5)
        
        # Add probability labels
        for i, (bar, prob) in enumerate(zip(bars, top_probs)):
            width = bar.get_width()
            ax.text(width + 0.02, bar.get_y() + bar.get_height()/2.,
                    f'{prob:.2f}', ha='left', va='center', fontsize=8, fontweight='bold')
        
        # Title with text
        title_text = text if len(text) <= 50 else text[:47] + "..."
        ax.set_title(f'"{title_text}"', fontsize=10, fontweight='bold', pad=10)
        ax.grid(True, alpha=0.3, axis='x')
    
    fig.suptitle('Example Predictions: Top 5 Categories per Query', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    plt.savefig('/mnt/user-data/outputs/4_example_predictions.png', dpi=300, bbox_inches='tight')
    print("✓ Created: 4_example_predictions.png")
    plt.close()


def diagram_5_model_architecture():
    """
    Create a simple diagram showing model architecture.
    """
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis('off')
    
    # Define boxes
    boxes = [
        {'name': 'User Input Text', 'pos': (0.5, 0.9), 'color': 'lightblue', 
         'example': '"I love beaches and nightlife"'},
        {'name': 'Tokenizer\n(DistilBERT)', 'pos': (0.5, 0.75), 'color': 'lightgreen'},
        {'name': 'DistilBERT Model\n(Fine-tuned)', 'pos': (0.5, 0.55), 'color': 'lightyellow',
         'details': '66M parameters\nMulti-label classification head'},
        {'name': '16 Output Neurons\n(Sigmoid Activation)', 'pos': (0.5, 0.35), 'color': 'lightcoral',
         'details': 'One for each interest category'},
        {'name': 'Predictions\n(Threshold = 0.5)', 'pos': (0.5, 0.15), 'color': 'lightgray',
         'example': 'beaches_positive: 0.89\nnightlife_positive: 0.76'}
    ]
    
    # Draw boxes and arrows
    for i, box in enumerate(boxes):
        # Draw box
        rect = plt.Rectangle((box['pos'][0] - 0.2, box['pos'][1] - 0.05), 
                            0.4, 0.1, facecolor=box['color'], 
                            edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        
        # Add name
        ax.text(box['pos'][0], box['pos'][1], box['name'], 
               ha='center', va='center', fontsize=11, fontweight='bold')
        
        # Add details or example
        if 'details' in box:
            ax.text(box['pos'][0], box['pos'][1] - 0.08, box['details'], 
                   ha='center', va='top', fontsize=8, style='italic')
        if 'example' in box:
            ax.text(box['pos'][0], box['pos'][1] - 0.08, box['example'], 
                   ha='center', va='top', fontsize=8, style='italic', color='blue')
        
        # Draw arrow to next box
        if i < len(boxes) - 1:
            ax.arrow(box['pos'][0], box['pos'][1] - 0.06, 
                    0, -0.08, head_width=0.03, head_length=0.02, 
                    fc='black', ec='black', linewidth=2)
    
    # Add title and info
    ax.text(0.5, 0.98, 'Model Architecture: Travel Interest Classification', 
           ha='center', va='top', fontsize=14, fontweight='bold')
    
    info_text = """Model Details:
• Base: DistilBERT (distilbert-base-uncased)
• Task: Multi-label classification
• Labels: 16 (8 positive + 8 negative interests)
• Training: 8 epochs, batch size 16
• Best F1: 97.8% (micro), 88.9% (sample)"""
    
    ax.text(0.05, 0.5, info_text, ha='left', va='center', fontsize=9,
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    
    plt.savefig('/mnt/user-data/outputs/5_model_architecture.png', dpi=300, bbox_inches='tight')
    print("✓ Created: 5_model_architecture.png")
    plt.close()


def generate_all_diagrams(data_path, model_path):
    """
    Generate all diagrams in one go.
    """
    print("=" * 60)
    print("GENERATING ALL VISUALIZATION DIAGRAMS")
    print("=" * 60)
    
    print("\n[1/5] Training curves...")
    diagram_1_training_curves()
    
    print("\n[2/5] Label distribution...")
    diagram_2_label_distribution(data_path)
    
    print("\n[3/5] Per-category performance (this may take a while)...")
    diagram_3_per_category_performance(model_path, data_path, is_validation=True)
    
    print("\n[4/5] Example predictions...")
    diagram_4_example_predictions(model_path)
    
    print("\n[5/5] Model architecture...")
    diagram_5_model_architecture()
    
    print("\n" + "=" * 60)
    print("ALL DIAGRAMS GENERATED SUCCESSFULLY!")
    print("=" * 60)
    print("\nFiles saved in: /mnt/user-data/outputs/")
    print("  1. 1_training_curves.png")
    print("  2. 2_label_distribution.png")
    print("  3. 3_per_category_performance.png")
    print("  4. 4_example_predictions.png")
    print("  5. 5_model_architecture.png")
    print("\nUse these diagrams to explain your model in presentations/reports.")


if __name__ == "__main__":
    # Paths - adjust these if needed
    DATA_PATH = "/mnt/datanew/nlc_dataset_new.jsonl"
    MODEL_PATH = "/mnt/datanew/fine_tuned_model"  # Your Modal volume path
    
    # If running locally (not on Modal), you might need to adjust paths:
    # DATA_PATH = "nlc_dataset_new.jsonl"
    # MODEL_PATH = "./fine_tuned_model"
    
    generate_all_diagrams(DATA_PATH, MODEL_PATH)

GENERATING ALL VISUALIZATION DIAGRAMS

[1/5] Training curves...
✓ Created: 1_training_curves.png

[2/5] Label distribution...
✓ Created: 2_label_distribution.png
  Total examples: 4206
  Most common: ('beaches_positive', 4206)
  Least common: ('religious_negative', 4206)

[3/5] Per-category performance (this may take a while)...
Evaluating on 842 examples...


/tmp/ipykernel_83/492784889.py:283: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


✓ Created: 3_per_category_performance.png
  Average F1: 0.127
  Categories with F1 > 95%: 0/16

Detailed per-category results:
  adventure_negative        - F1: 0.051, Precision: 1.000, Recall: 0.026, Support: 842
  adventure_positive        - F1: 0.091, Precision: 1.000, Recall: 0.048, Support: 842
  beaches_negative          - F1: 0.078, Precision: 1.000, Recall: 0.040, Support: 842
  beaches_positive          - F1: 0.129, Precision: 1.000, Recall: 0.069, Support: 842
  food_negative             - F1: 0.062, Precision: 1.000, Recall: 0.032, Support: 842
  food_positive             - F1: 0.110, Precision: 1.000, Recall: 0.058, Support: 842
  historical_negative       - F1: 0.051, Precision: 1.000, Recall: 0.026, Support: 842
  historical_positive       - F1: 0.086, Precision: 1.000, Recall: 0.045, Support: 842
  nature_negative           - F1: 0.035, Precision: 1.000, Recall: 0.018, Support: 842
  nature_positive           - F1: 0.093, Precision: 1.000, Recall: 0.049, Support: 842
  n

In [14]:
# Label Definition (Alphabetical Order)
LABELS = [
    'adventure_negative', 'adventure_positive', 'beaches_negative', 'beaches_positive',
    'food_negative', 'food_positive', 'historical_negative', 'historical_positive',
    'nature_negative', 'nature_positive', 'nightlife_negative', 'nightlife_positive',
    'religious_negative', 'religious_positive', 'shopping_negative', 'shopping_positive'
]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}
def predict_intent_debug(text):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DistilBertForSequenceClassification.from_pretrained(MODEL_SAVE_PATH).to(device)
    tokenizer = DistilBertTokenizer.from_pretrained(MODEL_SAVE_PATH)
    model.eval()
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.sigmoid(logits).squeeze().cpu().numpy() 
    
    print(f"Query: {text}")
    print("-" * 30)
    
    # Sort by probability to see what it's thinking
    top_indices = probs.argsort()[-5:][::-1] 
    
    for i in top_indices:
        print(f"{ID_TO_LABEL[i]}: {probs[i]:.4f}")
    
    #threshold
    print("-" * 30)
    print("Prediction with 0.5 threshold:")
    preds = (probs > 0.5)
    result = []
    for i, pred in enumerate(preds):
        if pred == 1:
            result.append(ID_TO_LABEL[i])
    print(result)

# Test
predict_intent_debug("I like beaches and forts, I don't like shopping")
predict_intent_debug("I love food and shopping, I hate nightlife")
predict_intent_debug("beaches")
predict_intent_debug("i dont like beaches and historical places")


Query: I like beaches and forts, I don't like shopping
------------------------------
beaches_positive: 0.9058
shopping_negative: 0.8910
historical_positive: 0.5526
beaches_negative: 0.0660
historical_negative: 0.0537
------------------------------
Prediction with 0.5 threshold:
['beaches_positive', 'historical_positive', 'shopping_negative']
Query: I love food and shopping, I hate nightlife
------------------------------
nightlife_negative: 0.8010
food_positive: 0.7190
shopping_positive: 0.7125
nightlife_positive: 0.0291
shopping_negative: 0.0233
------------------------------
Prediction with 0.5 threshold:
['food_positive', 'nightlife_negative', 'shopping_positive']
Query: beaches
------------------------------
beaches_positive: 0.9919
nightlife_positive: 0.0087
food_positive: 0.0085
nature_positive: 0.0084
shopping_positive: 0.0054
------------------------------
Prediction with 0.5 threshold:
['beaches_positive']
Query: i dont like beaches and historical places
---------------------

NameError: name 'plt' is not defined

In [11]:
print("Validation size:", len(val_texts))
print("Avg labels per sample:", label_array.sum(axis=1).mean())
print("Label density:", label_array.mean())

Validation size: 768


NameError: name 'label_array' is not defined

In [15]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Configuration ---
OUTPUT_DIR = '/mnt/user-data/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Updated Data from Training Logs ---
TRAINING_DATA = {
    'epochs': list(range(1, 9)), # 8 Epochs total
    'train_loss': [0.269000, 0.121400, 0.065500, 0.030900, 0.027200, 0.015100, 0.016200, 0.012100],
    'val_loss':   [0.245304, 0.107673, 0.055294, 0.033248, 0.023088, 0.018928, 0.016494, 0.014749],
    'f1_micro':   [0.096100, 0.870303, 0.958143, 0.984340, 0.989231, 0.987737, 0.987755, 0.986657],
    'f1_sample':  [0.034142, 0.879303, 0.960587, 0.987959, 0.991280, 0.989267, 0.989528, 0.989311]
}

def create_training_curves():
    plt.style.use('seaborn-v0_8-darkgrid')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = TRAINING_DATA['epochs']
    
    # Plot 1: Loss curves
    ax1.plot(epochs, TRAINING_DATA['train_loss'], 'b-o', label='Training Loss', 
             linewidth=2, markersize=6)
    ax1.plot(epochs, TRAINING_DATA['val_loss'], 'r-s', label='Validation Loss', 
             linewidth=2, markersize=6)
    
    # Mark Best Model at Epoch 5 (Peak F1 Score)
    ax1.axvline(x=5, color='green', linestyle='--', linewidth=2, 
                label='Best Model (Epoch 5)', alpha=0.7)
    
    ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Loss', fontsize=12, fontweight='bold')
    ax1.set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 9) # Adjusted for 8 epochs
    
    # Plot 2: F1 Scores
    ax2.plot(epochs, TRAINING_DATA['f1_micro'], 'g-o', label='F1 Micro', 
             linewidth=2, markersize=6)
    ax2.plot(epochs, TRAINING_DATA['f1_sample'], 'm-^', label='F1 Sample', 
             linewidth=2, markersize=6)
    
    ax2.axvline(x=5, color='green', linestyle='--', linewidth=2, 
                label='Best Model (Epoch 5)', alpha=0.7)
    ax2.axhline(y=0.98, color='orange', linestyle=':', linewidth=1.5, 
                label='98% Threshold', alpha=0.7)
    
    ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax2.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
    ax2.set_title('Model Performance (F1 Scores)', fontsize=14, fontweight='bold')
    ax2.legend(loc='lower right')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, 9)
    ax2.set_ylim(0, 1.05)
    
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, '1_training_curves.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Created: {save_path}")
    plt.close()

if __name__ == "__main__":
    create_training_curves()

✓ Created: /mnt/user-data/outputs/1_training_curves.png


In [16]:


# --- Configuration ---
OUTPUT_DIR = '/mnt/user-data/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def create_architecture_diagram():
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis('off')
    
    # Define boxes
    boxes = [
        {'name': 'User Input Text', 'pos': (0.5, 0.9), 'color': 'lightblue', 
         'example': '"I love beaches and nightlife"'},
        {'name': 'Tokenizer\n(DistilBERT)', 'pos': (0.5, 0.75), 'color': 'lightgreen'},
        {'name': 'DistilBERT Model\n(Fine-tuned)', 'pos': (0.5, 0.55), 'color': 'lightyellow',
         'details': '66M parameters\nMulti-label classification head'},
        {'name': '16 Output Neurons\n(Sigmoid Activation)', 'pos': (0.5, 0.35), 'color': 'lightcoral',
         'details': 'One for each interest category'},
        {'name': 'Predictions\n(Threshold = 0.5)', 'pos': (0.5, 0.15), 'color': 'lightgray',
         'example': 'beaches_positive: 0.89\nnightlife_positive: 0.76'}
    ]
    
    # Draw boxes and arrows
    for i, box in enumerate(boxes):
        rect = plt.Rectangle((box['pos'][0] - 0.2, box['pos'][1] - 0.05), 
                            0.4, 0.1, facecolor=box['color'], 
                            edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        
        ax.text(box['pos'][0], box['pos'][1], box['name'], 
               ha='center', va='center', fontsize=11, fontweight='bold')
        
        if 'details' in box:
            ax.text(box['pos'][0], box['pos'][1] - 0.08, box['details'], 
                   ha='center', va='top', fontsize=8, style='italic')
        if 'example' in box:
            ax.text(box['pos'][0], box['pos'][1] - 0.08, box['example'], 
                   ha='center', va='top', fontsize=8, style='italic', color='blue')
        
        if i < len(boxes) - 1:
            ax.arrow(box['pos'][0], box['pos'][1] - 0.06, 
                    0, -0.08, head_width=0.03, head_length=0.02, 
                    fc='black', ec='black', linewidth=2)
    
    ax.text(0.5, 0.98, 'Model Architecture: Travel Interest Classification', 
           ha='center', va='top', fontsize=14, fontweight='bold')
    
    # Updated statistics text
    info_text = """Model Details:
• Base: DistilBERT (distilbert-base-uncased)
• Task: Multi-label classification
• Labels: 16 (8 positive + 8 negative interests)
• Training: 5 epochs (Best Checkpoint)
• Best F1: 98.9% (micro), 99.1% (sample)"""
    
    ax.text(0.05, 0.5, info_text, ha='left', va='center', fontsize=9,
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    
    save_path = os.path.join(OUTPUT_DIR, '5_model_architecture.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Created: {save_path}")
    plt.close()

if __name__ == "__main__":
    create_architecture_diagram()

✓ Created: /mnt/user-data/outputs/5_model_architecture.png


In [17]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import os
from collections import Counter
from matplotlib.patches import Patch

# --- Configuration ---
OUTPUT_DIR = '/mnt/user-data/outputs'
DATA_PATH = "/mnt/datanew/nlc_dataset_new.jsonl"  # Update path if needed
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABELS = [
    'adventure_negative', 'adventure_positive', 'beaches_negative', 'beaches_positive',
    'food_negative', 'food_positive', 'historical_negative', 'historical_positive',
    'nature_negative', 'nature_positive', 'nightlife_negative', 'nightlife_positive',
    'religious_negative', 'religious_positive', 'shopping_negative', 'shopping_positive'
]

def create_label_distribution():
    plt.style.use('seaborn-v0_8-darkgrid')
    
    # Load data
    data = []
    print(f"Loading data from {DATA_PATH}...")
    with open(DATA_PATH, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    
    # Count label occurrences
    label_counts = Counter()
    for item in data:
        for label in item['labels']:
            label_counts[label] += 1
    
    # Sort labels for better visualization
    sorted_labels = sorted(LABELS)
    counts = [label_counts[label] for label in sorted_labels]
    
    # Create colors
    colors = ['green' if 'positive' in label else 'red' for label in sorted_labels]
    
    # Create plot
    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.bar(range(len(sorted_labels)), counts, color=colors, alpha=0.7, 
                  edgecolor='black', linewidth=1)
    
    # Add value labels on bars
    for i, (bar, count) in enumerate(zip(bars, counts)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 5,
                f'{count}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Format labels
    clean_labels = [label.replace('_', ' ').title() for label in sorted_labels]
    ax.set_xticks(range(len(sorted_labels)))
    ax.set_xticklabels(clean_labels, rotation=45, ha='right', fontsize=10)
    ax.set_xlabel('Interest Category', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Examples', fontsize=12, fontweight='bold')
    ax.set_title('Distribution of Labels in Training Dataset', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add legend
    legend_elements = [
        Patch(facecolor='green', alpha=0.7, label='Positive Interest'),
        Patch(facecolor='red', alpha=0.7, label='Negative Interest')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    # Add statistics
    total = len(data)
    avg_labels_per_example = sum(len(item['labels']) for item in data) / total
    stats_text = f"Total Examples: {total}\nAvg Labels/Example: {avg_labels_per_example:.2f}"
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, '2_label_distribution.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Created: {save_path}")
    plt.close()

if __name__ == "__main__":
    create_label_distribution()

Loading data from /mnt/datanew/nlc_dataset_new.jsonl...
✓ Created: /mnt/user-data/outputs/2_label_distribution.png


In [22]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

# --- Configuration ---
OUTPUT_DIR = '/mnt/user-data/outputs'
DATA_PATH = "/mnt/datanew/nlc_dataset_new.jsonl"
MODEL_PATH = "/mnt/datanew/fine_tuned_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABELS = [
    'adventure_negative', 'adventure_positive', 'beaches_negative', 'beaches_positive',
    'food_negative', 'food_positive', 'historical_negative', 'historical_positive',
    'nature_negative', 'nature_positive', 'nightlife_negative', 'nightlife_positive',
    'religious_negative', 'religious_positive', 'shopping_negative', 'shopping_positive'
]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}

def create_performance_diagram():
    plt.style.use('seaborn-v0_8-darkgrid')
    
    # Load model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Loading model from {MODEL_PATH} on {device}...")
    model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
    tokenizer = DistilBertTokenizer.from_pretrained(MODEL_PATH)
    model.eval()
    
    # Load data
    data = []
    with open(DATA_PATH, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    
    # Use last 20% as validation
    split_idx = int(len(data) * 0.8)
    val_data = data[split_idx:]
    print(f"Evaluating on {len(val_data)} examples...")

    all_preds = []
    all_labels = []
    
    # --- SMART LABEL EXTRACTION V2 ---
    if len(val_data) > 0:
        raw_sample = val_data[0].get('labels', [])
        print(f"DEBUG: Found label structure: {str(raw_sample)[:200]}...") # Show sample
        
        # Case 1: Dictionary (e.g., {"adventure_positive": 1})
        if isinstance(raw_sample, dict):
            print("INFO: Detected DICTIONARY. Checking keys vs values...")
            # Check if KEYS match our label list
            sample_keys = list(raw_sample.keys())
            sample_values = list(raw_sample.values())
            
            matches_in_keys = sum(1 for k in sample_keys if k in LABEL_TO_ID)
            matches_in_values = sum(1 for v in sample_values if v in LABEL_TO_ID)
            
            if matches_in_keys > 0:
                print("INFO: Label names found in DICTIONARY KEYS. Using keys.")
                label_format = 'dict_keys'
            elif matches_in_values > 0:
                print("INFO: Label names found in DICTIONARY VALUES. Using values.")
                label_format = 'dict_values'
            else:
                print("WARNING: Could not match dictionary keys or values to label list!")
                print(f"DEBUG: First few keys: {sample_keys[:5]}")
                label_format = 'unknown'

        # Case 2: List of Integers
        elif isinstance(raw_sample, list) and len(raw_sample) > 0 and isinstance(raw_sample[0], int):
            print("INFO: Detected INTEGER LIST format.")
            label_format = 'int_list'
            
        # Case 3: List of Strings
        else:
            print("INFO: Detected STRING LIST format.")
            label_format = 'str_list'

    for item in val_data:
        text = item['text']
        true_labels = [0] * len(LABELS)
        raw_labels = item.get('labels', [])
        
        if label_format == 'dict_keys':
            # Use keys where value is truthy (e.g., {"label": 1})
            for key, val in raw_labels.items():
                if val == 1 or val is True: # Check if label is active
                    if key in LABEL_TO_ID:
                        true_labels[LABEL_TO_ID[key]] = 1
                        
        elif label_format == 'dict_values':
            # Use values (rare, but possible)
            for l in raw_labels.values():
                 if l in LABEL_TO_ID:
                    true_labels[LABEL_TO_ID[l]] = 1
                    
        elif label_format == 'int_list':
            for idx in raw_labels:
                if 0 <= idx < len(LABELS):
                    true_labels[idx] = 1
                    
        elif label_format == 'str_list':
            for l in raw_labels:
                if l in LABEL_TO_ID:
                    true_labels[LABEL_TO_ID[l]] = 1
        
        # Model Inference
        inputs = tokenizer(text, return_tensors="pt", truncation=True, 
                          padding=True, max_length=128).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.sigmoid(logits).squeeze().cpu().numpy()
        
        preds = (probs > 0.5).astype(int)
        all_preds.append(preds)
        all_labels.append(true_labels)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Calculate metrics
    category_metrics = []
    for i, label in enumerate(LABELS):
        pred_col = all_preds[:, i]
        true_col = all_labels[:, i]
        
        tp = ((pred_col == 1) & (true_col == 1)).sum()
        fp = ((pred_col == 1) & (true_col == 0)).sum()
        fn = ((pred_col == 0) & (true_col == 1)).sum()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        category_metrics.append({'label': label, 'f1': f1})
    
    # Plotting
    fig, ax = plt.subplots(figsize=(14, 7))
    
    f1_scores = [m['f1'] for m in category_metrics]
    labels = [m['label'] for m in category_metrics]
    colors = ['green' if 'positive' in label else 'red' for label in labels]
    
    bars = ax.bar(range(len(labels)), f1_scores, color=colors, alpha=0.7, edgecolor='black')
    
    ax.axhline(y=0.95, color='orange', linestyle='--', linewidth=2, label='95% Threshold')
    ax.axhline(y=0.90, color='yellow', linestyle=':', linewidth=1.5, label='90% Threshold')
    ax.set_ylim(0, 1.05)
    
    clean_labels = [label.replace('_', ' ').title() for label in labels]
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(clean_labels, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
    ax.set_title('Per-Category Model Performance', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(loc='lower right')
    
    avg_f1 = np.mean(f1_scores)
    stats_text = f"Average F1: {avg_f1:.3f}"
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, '3_per_category_performance.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Created: {save_path}")
    print(f"Average F1 Score: {avg_f1:.4f}")
    plt.close()

if __name__ == "__main__":
    create_performance_diagram()

Loading model from /mnt/datanew/fine_tuned_model on cuda...
Evaluating on 842 examples...
DEBUG: Found label structure: {'beaches_positive': 0, 'beaches_negative': 0, 'historical_positive': 0, 'historical_negative': 0, 'adventure_positive': 1, 'adventure_negative': 0, 'nature_positive': 0, 'nature_negative': 0, 'food_p...
INFO: Detected DICTIONARY. Checking keys vs values...
INFO: Label names found in DICTIONARY KEYS. Using keys.
✓ Created: /mnt/user-data/outputs/3_per_category_performance.png
Average F1 Score: 0.9931


In [19]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

# --- Configuration ---
OUTPUT_DIR = '/mnt/user-data/outputs'
MODEL_PATH = "/mnt/datanew/fine_tuned_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABELS = [
    'adventure_negative', 'adventure_positive', 'beaches_negative', 'beaches_positive',
    'food_negative', 'food_positive', 'historical_negative', 'historical_positive',
    'nature_negative', 'nature_positive', 'nightlife_negative', 'nightlife_positive',
    'religious_negative', 'religious_positive', 'shopping_negative', 'shopping_positive'
]

def create_example_predictions():
    plt.style.use('seaborn-v0_8-darkgrid')
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Loading model from {MODEL_PATH}...")
    model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
    tokenizer = DistilBertTokenizer.from_pretrained(MODEL_PATH)
    model.eval()
    
    test_examples = [
        "I love exploring ancient temples and historical sites",
        "I enjoy beach parties and nightlife but hate museums",
        "Looking for adventure activities like hiking and camping",
        "I prefer quiet nature walks, not interested in shopping or nightlife",
        "I want to try local food and visit markets",
        "Beach resorts are boring, I prefer mountain trekking"
    ]
    
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.4, wspace=0.3)
    
    for idx, text in enumerate(test_examples):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        
        inputs = tokenizer(text, return_tensors="pt", truncation=True, 
                          padding=True, max_length=128).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.sigmoid(logits).squeeze().cpu().numpy()
        
        # Top 5 predictions
        top_indices = probs.argsort()[-5:][::-1]
        top_labels = [LABELS[i].replace('_', ' ').title() for i in top_indices]
        top_probs = [probs[i] for i in top_indices]
        top_colors = ['green' if 'Positive' in label else 'red' for label in top_labels]
        
        bars = ax.barh(range(5), top_probs, color=top_colors, alpha=0.7, edgecolor='black')
        ax.set_yticks(range(5))
        ax.set_yticklabels(top_labels, fontsize=9)
        ax.set_xlim(0, 1)
        ax.set_xlabel('Confidence Score', fontsize=9)
        ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1, alpha=0.5)
        
        title_text = text if len(text) <= 50 else text[:47] + "..."
        ax.set_title(f'"{title_text}"', fontsize=10, fontweight='bold', pad=10)
        ax.grid(True, alpha=0.3, axis='x')
    
    fig.suptitle('Example Predictions: Top 5 Categories per Query', fontsize=16, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, '4_example_predictions.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Created: {save_path}")
    plt.close()

if __name__ == "__main__":
    create_example_predictions()

Loading model from /mnt/datanew/fine_tuned_model...


/tmp/ipykernel_123/4191804899.py:70: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


✓ Created: /mnt/user-data/outputs/4_example_predictions.png


In [23]:
f1_score(y_true.flatten(), y_pred.flatten())

NameError: name 'f1_score' is not defined